# Entrena tu propio recomendador generativo (mini-TIGER)
**IIC3633 Sistemas Recomendadores · PUC Chile · Módulo "Recomendadores Generativos"**

En este notebook recorrerás, de principio a fin y en una GPU gratuita de Colab (T4), el pipeline de un recomendador generativo basado en **Semantic IDs**:

1. **Texto del ítem → embedding** (encoder de `flan-t5-base`).
2. **Embedding → Semantic ID** con **RQ-KMeans** (cuantización residual sin red neuronal) + token extra para resolver colisiones.
3. **Entrenar un Transformer encoder-decoder (T5 pequeño, estilo TIGER)** que, dada la historia de un usuario (secuencia de SIDs), *genera* el SID del siguiente ítem token a token.
4. **Inferencia con beam search restringido** (trie) para que solo se generen SIDs válidos.
5. **Evaluar** con Recall@K y NDCG@K (leave-one-out) y comparar con un baseline de popularidad.
6. **Cambiar el tokenizador por un RQ-VAE** (el de TIGER) y comparar ambos.

**Referencia:** este notebook es una versión autocontenida e inspirada en el repositorio [snap-research/GRID](https://github.com/snap-research/GRID) (*Generative Recommendation with Semantic IDs*, Ju et al., 2025), que usa Hydra + PyTorch Lightning y embeddings `flan-t5-xl`. Aquí simplificamos (PyTorch + HuggingFace) para que corra en minutos. Si quieres reproducir los resultados "oficiales", usa el repositorio original.

Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

> Los ejercicios están marcados como **🧪 Ejercicio**. Hay una solución de referencia para el ejercicio de código, pero inténtalo primero.

# Parte A
## 0. Preparación

In [ ]:
!pip -q install transformers sentencepiece
import os, gzip, json, ast, math, time, random, urllib.request, collections, copy
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)

# SYNTHETIC=1 solo se usa para probar el notebook sin internet (no lo cambies como estudiante)
SYNTHETIC = os.environ.get("GENREC_SYNTHETIC", "0") == "1"

CFG = dict(
    category="Musical_Instruments",
    emb_model="google/flan-t5-base",   # GRID usa flan-t5-xl (2048-d); base (768-d) es suficiente aquí
    levels=3, K=256,                   # 3 niveles x 256 códigos (como TIGER) + 1 token anti-colisión
    max_items=10,                      # largo máximo de la historia (en ítems); TIGER usa 20, aquí 10 para ir ~2x más rápido
    d_model=128, d_ff=1024, heads=6, d_kv=64, layers=4, dropout=0.1,   # T5 pequeño tipo TIGER
    lr=1e-3, batch=256, epochs=10, patience=3, val_users=1000,
    use_amp=False,                     # True = precisión mixta fp16 (más rápido en la T4, ver sección 6)
    beams=20, topk=10,
    test_n=None,                       # None = evaluar en todos los usuarios de test (~4–5 min); p. ej. 5000 para ir más rápido
)
if SYNTHETIC:
    CFG.update(K=16, max_items=8, d_model=64, d_ff=128, heads=2, d_kv=32, layers=2, epochs=6, batch=64, beams=10)
CFG

## 1. Datos: Amazon Reviews — *Beauty* (5-core)
Usamos el mismo conjunto que GRID/P5/TIGER: ~22k usuarios, ~12k ítems, ~200k interacciones. Cada usuario es una secuencia de ítems ordenada por tiempo. Partición **leave-one-out**: el último ítem es *test*, el penúltimo *validación* y el resto *entrenamiento*.

El texto de cada ítem (título, marca, categorías) será lo único que usemos para construir su Semantic ID: **el modelo nunca ve el id numérico del ítem**.

> Si la descarga falla (los servidores académicos a veces están caídos), sube manualmente `reviews_Beauty_5.json.gz` y `meta_Beauty.json.gz` a la carpeta `/content/data/` (panel izquierdo de Colab) y vuelve a ejecutar la celda.

In [ ]:
DATA_DIR = "/content/data" if os.path.isdir("/content") else "./data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE = ["https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/",
        "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/",
        "https://jmcauley.ucsd.edu/data/amazon/categoryFiles/"]

CATEGORY = CFG["category"]   # nombre tal como aparece en los archivos de 2014

def download(fname):
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest): return dest
    for b in BASE:
        try:
            print("descargando", b + fname)
            urllib.request.urlretrieve(b + fname, dest); return dest
        except Exception as e:
            print("  falló:", e)
    raise RuntimeError(f"No se pudo descargar {fname}. Súbelo manualmente a {DATA_DIR}.")

def load_amazon(category=CATEGORY):
    rev = download(f"reviews_{category}_5.json.gz")
    meta = download(f"meta_{category}.json.gz")
    inter = collections.defaultdict(list)
    with gzip.open(rev, "rt") as f:
        for line in f:
            r = json.loads(line)
            inter[r["reviewerID"]].append((r["unixReviewTime"], r["asin"]))
    asins = sorted({a for v in inter.values() for _, a in v})
    asin2id = {a: i for i, a in enumerate(asins)}
    info = {}
    with gzip.open(meta, "rt") as f:
        for line in f:
            a = line.split("'")[3]                      # los metadatos 2014 son dicts de Python, uno por línea
            if a in asin2id:
                d = ast.literal_eval(line)
                cats = " > ".join(d.get("categories", [[]])[0]) if d.get("categories") else ""
                info[a] = (d.get("title", ""), d.get("brand", ""), cats)
    items = []
    for a in asins:
        t, b, c = info.get(a, ("", "", ""))
        items.append({"asin": a, "title": t or a, "text": f"Title: {t}. Brand: {b}. Categories: {c}."})
    seqs = [[asin2id[a] for _, a in sorted(v)] for v in inter.values()]
    return items, seqs

def make_synthetic(n_items=600, n_users=800, n_cl=12, seed=0):
    # Datos de juguete: ítems en grupos y usuarios que tienden a quedarse dentro de un grupo.
    rng = np.random.RandomState(seed)
    cl = rng.randint(0, n_cl, n_items)
    items = [{"asin": f"S{i}", "title": f"ítem {i} (grupo {cl[i]})",
              "text": f"Title: ítem {i}. Categories: grupo {cl[i]}."} for i in range(n_items)]
    by_cl = [np.where(cl == c)[0] for c in range(n_cl)]
    seqs = []
    for _ in range(n_users):
        c, s = rng.randint(n_cl), []
        for _ in range(rng.randint(5, 15)):
            if rng.rand() < 0.2: c = rng.randint(n_cl)
            s.append(int(rng.choice(by_cl[c])))
        seqs.append(s)
    return items, seqs, cl

if SYNTHETIC:
    items, seqs, _syn_cl = make_synthetic()
else:
    items, seqs = load_amazon()
N = len(items)
print(f"{len(seqs)} usuarios, {N} ítems, {sum(map(len, seqs))} interacciones")
print("Ejemplo de ítem:", items[0]["text"][:200])

## 2. Embeddings de texto con `flan-t5-base`
Codificamos el texto de cada ítem con el *encoder* de Flan-T5 y promediamos los estados ocultos (mean pooling) → un vector de 768 dimensiones por ítem. (TIGER usa Sentence-T5; GRID usa `flan-t5-xl`.)

In [ ]:
@torch.no_grad()
def embed_texts(texts, model_name, bs=128, max_len=128):
    from transformers import AutoTokenizer, T5EncoderModel
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = T5EncoderModel.from_pretrained(model_name).to(device).eval()
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).float().cpu())
    del enc; torch.cuda.empty_cache() if device == "cuda" else None
    return torch.cat(out).numpy()

t0 = time.time()
if SYNTHETIC:
    rng = np.random.RandomState(1)
    centers = rng.randn(12, 64) * 3
    emb = (centers[_syn_cl] + rng.randn(N, 64)).astype(np.float32)
else:
    emb = embed_texts([it["text"] for it in items], CFG["emb_model"])
print("embeddings:", emb.shape, f"({time.time()-t0:.0f}s)")

In [ ]:
# Verificación rápida: vecinos más cercanos (coseno) de un ítem según su embedding
e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
q = 123 % N
sims = e @ e[q]
print("ÍTEM:", items[q]["title"][:90])
for j in np.argsort(-sims)[1:6]:
    print(f"  {sims[j]:.3f}  {items[j]['title'][:90]}")

## 3. Semantic IDs con RQ-KMeans
La idea es la **cuantización residual** (la misma del RQ-VAE de TIGER, pero sin red neuronal):

- Nivel 1: k-means sobre los embeddings → cada ítem recibe el código del centroide más cercano, $c_1$.
- Nivel 2: k-means sobre el **residuo** $r_1 = x - \mu_{c_1}$ → código $c_2$.
- Nivel 3: ídem sobre $r_2 = r_1 - \mu_{c_2}$ → código $c_3$.

El Semantic ID es $(c_1, c_2, c_3)$: de **grueso a fino**. Cada nivel corrige lo que el anterior no alcanzó a explicar, por lo que ítems parecidos comparten prefijo. 

![Diagrama RQ-KMeans](https://raw.githubusercontent.com/PUC-RecSys-Class/RecSysPUC-2026-2/master/practicos/RQ-KMeans.png)

In [ ]:
def kmeans(x, k, iters=30, seed=0):
    g = torch.Generator(device=x.device).manual_seed(seed)
    c = x[torch.randperm(x.size(0), generator=g, device=x.device)[:k]].clone()
    for _ in range(iters):
        a = torch.cdist(x, c).argmin(1)
        s = torch.zeros_like(c).index_add_(0, a, x)
        cnt = torch.bincount(a, minlength=k).float()
        new = s / cnt.clamp(min=1).unsqueeze(1)
        new[cnt == 0] = c[cnt == 0]                  # clusters vacíos: se conservan
        c = new
    a = torch.cdist(x, c).argmin(1)
    return c, a

def rq_kmeans(emb, levels, K):
    x = torch.tensor(emb, device=device)
    res, codes, errs = x.clone(), [], []
    for l in range(levels):
        c, a = kmeans(res, K, seed=l)
        res = res - c[a]                              # residuo para el siguiente nivel
        codes.append(a.cpu())
        errs.append((res.norm(dim=1).mean() / x.norm(dim=1).mean()).item())
    return torch.stack(codes, 1).numpy(), errs

t0 = time.time()
codes, errs = rq_kmeans(emb, CFG["levels"], CFG["K"])
print(f"RQ-KMeans listo en {time.time()-t0:.0f}s")
print("error relativo de reconstrucción tras cada nivel:", [f"{e:.3f}" for e in errs])
print("SID (3 niveles) del ítem", q, "->", codes[q])

### 🧪 Ejercicio 1 — Resolver colisiones
Con solo 3 niveles, algunos ítems distintos pueden terminar con el **mismo** $(c_1,c_2,c_3)$ (colisión). TIGER añade un **cuarto token** para desambiguar: si $n$ ítems comparten los 3 primeros códigos, reciben 0, 1, …, $n-1$ como cuarto código.

Implementa `dedup_token(codes)`: recibe un arreglo `[N, 3]` y devuelve `[N, 4]` con el token extra. *Pista: un diccionario que cuente cuántas veces ya apareció cada tupla.*

In [ ]:
def dedup_token(codes):
    # TODO: devuelve un arreglo [N, 4]: las 3 columnas originales + el índice de desambiguación
    raise NotImplementedError("Implementa dedup_token")

In [ ]:
# Solución de referencia (úsala solo si te atascas). Si no implementaste el ejercicio, se usará esta.
def dedup_token_solution(codes):
    seen = collections.defaultdict(int); extra = []
    for row in map(tuple, codes):
        extra.append(seen[row]); seen[row] += 1
    return np.concatenate([codes, np.array(extra)[:, None]], 1)

try:
    sids = dedup_token(codes)
except NotImplementedError:
    print("⚠️ Usando la solución de referencia.")
    sids = dedup_token_solution(codes)

def collision_report(codes3):
    cnt = collections.Counter(map(tuple, codes3))
    n_coll = sum(c for c in cnt.values() if c > 1)
    print(f"ítems en colisión (mismo SID de 3 niveles): {n_coll}/{len(codes3)} ({100*n_coll/len(codes3):.1f}%) · "
          f"SIDs distintos: {len(cnt)} · máximo ítems por SID: {max(cnt.values())}")
collision_report(codes)
assert len(set(map(tuple, sids))) == N, "Los SIDs de 4 tokens deben ser únicos"
assert sids[:, 3].max() < CFG["K"], "Hay demasiadas colisiones para K"
D = sids.shape[1]; K = CFG["K"]
print("SID final del ítem", q, "->", sids[q])

In [ ]:
# ¿Los SIDs capturan semántica? Ítems que comparten (c1, c2) con el ítem q:
same = np.where((sids[:, 0] == sids[q, 0]) & (sids[:, 1] == sids[q, 1]))[0][:6]
print("ÍTEM:", items[q]["title"][:80], sids[q])
for j in same: print("  comparte (c1,c2):", items[j]["title"][:80], sids[j])
other = np.where(sids[:, 0] != sids[q, 0])[0][:3]
for j in other: print("  c1 distinto:      ", items[j]["title"][:80], sids[j])

## 4. Del SID al vocabulario del Transformer
Cada (nivel, código) se convierte en un **token** distinto: tokens `0`=PAD y `1`=EOS, y luego `2 + nivel·K + código`. Una historia de $m$ ítems se vuelve una secuencia de $4m$ tokens; el modelo debe generar los 4 tokens del siguiente ítem y luego EOS.

Para generar solo SIDs **válidos** construimos un **trie** con los SIDs existentes: en cada paso, el beam search solo puede elegir tokens que mantengan el prefijo dentro del trie.

In [ ]:
# ---------- Código común: tokens de SID, trie, recomendación y métricas ----------
PAD, EOS = 0, 1   # T5: pad=0 (también token de inicio del decoder), eos=1

def codes_to_tokens(codes, K):
    # codes [N, D] (cada nivel con valores 0..K-1) -> ids de token.
    # El nivel l usa el rango [2 + l*K, 2 + (l+1)*K): así cada (nivel, código) es un token distinto.
    D = codes.shape[1]
    return (codes + 2 + np.arange(D)[None, :] * K).astype(np.int64)

def vocab_size(D, K):
    return 2 + D * K

def build_trie(sid_tok):
    # prefijo (tupla de tokens) -> lista de tokens permitidos a continuación
    trie = collections.defaultdict(set)
    for row in sid_tok:
        row = tuple(int(t) for t in row)
        for k in range(len(row)):
            trie[row[:k]].add(row[k])
        trie[row] = {EOS}                      # SID completo: solo se puede cerrar
    return {k: sorted(v) for k, v in trie.items()}

def make_prefix_fn(trie):
    # Función que HuggingFace llama en cada paso del beam search.
    def fn(batch_id, input_ids):
        prefix = tuple(input_ids.tolist()[1:])  # se descarta el token de inicio del decoder
        return trie.get(prefix, [PAD])          # beams ya terminados: solo PAD
    return fn

def make_sid2item(sid_tok):
    return {tuple(int(t) for t in row): i for i, row in enumerate(sid_tok)}

def history_tokens(hist_items, sid_tok, max_items):
    hist = hist_items[-max_items:]
    return [int(t) for i in hist for t in sid_tok[i]]

@torch.no_grad()
def recommend(model, histories, sid_tok, trie, sid2item, max_items,
              beams=20, topk=10, bs=128, constrained=True):
    # Devuelve (items, scores): para cada historia, topk ids de ítem (-1 si el SID generado no existe).
    model.eval()
    D = sid_tok.shape[1]
    fn = make_prefix_fn(trie) if constrained else None
    all_items, all_scores = [], []
    for s in range(0, len(histories), bs):
        chunk = histories[s:s + bs]
        seqs = [history_tokens(h, sid_tok, max_items) for h in chunk]
        m = max(len(x) for x in seqs)
        ids = torch.zeros(len(seqs), m, dtype=torch.long)
        att = torch.zeros_like(ids)
        for i, x in enumerate(seqs):
            ids[i, :len(x)] = torch.tensor(x)
            att[i, :len(x)] = 1
        gen = model.generate(input_ids=ids.to(model.device), attention_mask=att.to(model.device),
                             num_beams=beams, num_return_sequences=topk, max_new_tokens=D + 1,
                             prefix_allowed_tokens_fn=fn, do_sample=False,
                             return_dict_in_generate=True, output_scores=True)
        seq = gen.sequences[:, 1:1 + D].cpu().tolist()
        sc = gen.sequences_scores.cpu().tolist()
        for i in range(len(chunk)):
            all_items.append([sid2item.get(tuple(seq[i * topk + j]), -1) for j in range(topk)])
            all_scores.append([sc[i * topk + j] for j in range(topk)])
    return all_items, all_scores

def ranking_metrics(ranked, targets, ks=(5, 10)):
    res = {f"{n}@{k}": 0.0 for k in ks for n in ("Recall", "NDCG")}
    for r, t in zip(ranked, targets):
        for k in ks:
            top = r[:k]
            if t in top:
                res[f"Recall@{k}"] += 1
                res[f"NDCG@{k}"] += 1 / math.log2(top.index(t) + 2)
    return {k: v / len(targets) for k, v in res.items()}

In [ ]:
sid_tok = codes_to_tokens(sids, K)
V = vocab_size(D, K)
trie = build_trie(sid_tok)
sid2item = make_sid2item(sid_tok)
print("tamaño de vocabulario:", V, "· nodos del trie:", len(trie))
print("tokens del ítem", q, ":", sid_tok[q].tolist())

## 5. Datos de entrenamiento y modelo
Para cada usuario y cada posición $t \ge 1$ de su secuencia de **entrenamiento** creamos un ejemplo: *historia* (ítems anteriores, hasta `max_items`) → *objetivo* (ítem $t$). Se entrena con **next-token prediction** (entropía cruzada) sobre los 4 tokens del SID + EOS.

El modelo es un **T5** (Transformer encoder-decoder) pequeño inicializado **desde cero**, con las dimensiones de TIGER (4 capas, 6 cabezas, $d_{model}=128$):

- El **encoder** lee la historia completa (4 tokens por ítem) con atención bidireccional: cada token puede mirar a todos los demás.
- El **decoder** genera el SID del siguiente ítem **de a un token**: $c_1$, luego $c_2$ dado $c_1$, etc. En cada paso mira lo que ya generó (atención causal) y la historia codificada (*cross-attention*).
- En inferencia, el **trie** de la sección 4 restringe cada paso del beam search a tokens que siguen formando un SID existente.

Como el SID es jerárquico, generar $c_1$ equivale a elegir una "categoría gruesa" y los tokens siguientes van refinando la elección. Para el entrenamiento solo tienes que entender el flujo; la función `fit` hace todo.

![Diagrama TIGER](https://raw.githubusercontent.com/PUC-RecSys-Class/RecSysPUC-2026-2/master/practicos/TIGER.png)

In [ ]:
from transformers import T5Config, T5ForConditionalGeneration

train_seqs = [s[:-2] for s in seqs if len(s) >= 3]
val_users  = [(s[:-2], s[-2]) for s in seqs if len(s) >= 3]    # historia, objetivo (validación)
test_users = [(s[:-1], s[-1]) for s in seqs if len(s) >= 3]    # historia, objetivo (test)

def build_train_examples(train_seqs, max_items):
    ex = []
    for s in train_seqs:
        for t in range(1, len(s)):
            ex.append((s[max(0, t - max_items):t], s[t]))
    return ex
train_ex = build_train_examples(train_seqs, CFG["max_items"])
print(f"{len(train_ex)} ejemplos de entrenamiento · {len(val_users)} usuarios val/test")

def make_collate(sid_tok, max_items):
    def collate(batch):
        seqs_ = [history_tokens(h, sid_tok, max_items) for h, _ in batch]
        m = max(len(x) for x in seqs_)
        ids = torch.zeros(len(batch), m, dtype=torch.long); att = torch.zeros_like(ids)
        for i, x in enumerate(seqs_):
            ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1
        labels = torch.tensor([[int(t) for t in sid_tok[tgt]] + [EOS] for _, tgt in batch])
        return ids, att, labels
    return collate

def new_model(V):
    cfg = T5Config(vocab_size=V, d_model=CFG["d_model"], d_ff=CFG["d_ff"], d_kv=CFG["d_kv"],
                   num_heads=CFG["heads"], num_layers=CFG["layers"], num_decoder_layers=CFG["layers"],
                   dropout_rate=CFG["dropout"], feed_forward_proj="relu",
                   pad_token_id=PAD, eos_token_id=EOS, decoder_start_token_id=PAD)
    return T5ForConditionalGeneration(cfg).to(device)

m_ = new_model(V)
print(f"parámetros del modelo: {sum(p.numel() for p in m_.parameters())/1e6:.1f} M"); del m_

## 6. Entrenamiento
Optimizador AdamW con *warm-up* y decaimiento coseno. Después de cada época se mide **Recall@10 en validación** (con beam search restringido sobre 1 000 usuarios) y nos quedamos con el mejor modelo (*early stopping*). En una T4 son ~1–2 min por época (la salida muestra el tiempo de cada época y el acumulado).

**Opcional — precisión mixta:** si pones `CFG["use_amp"] = True` (celda de configuración) el entrenamiento usa números de 16 bits (fp16) en la GPU, lo que debería acelerarlo bastante en la T4. Es una *mejora opcional*: si ves `loss nan` o que la pérdida no baja, vuelve a `False`. La evaluación siempre se hace en precisión completa.

In [ ]:
def evaluate(model, users, sid_tok, trie, sid2item, n=None, beams=None, topk=None, constrained=True, seed=0):
    users = list(users)
    if n and n < len(users):
        users = random.Random(seed).sample(users, n)
    ranked, _ = recommend(model, [h for h, _ in users], sid_tok, trie, sid2item, CFG["max_items"],
                          beams=beams or CFG["beams"], topk=topk or CFG["topk"], constrained=constrained)
    return ranking_metrics(ranked, [t for _, t in users])

def fit(sid_tok, epochs=None, verbose=True):
    epochs = epochs or CFG["epochs"]
    trie_, s2i_ = build_trie(sid_tok), make_sid2item(sid_tok)
    model = new_model(vocab_size(sid_tok.shape[1], K))
    loader = torch.utils.data.DataLoader(train_ex, batch_size=CFG["batch"], shuffle=True,
                                         collate_fn=make_collate(sid_tok, CFG["max_items"]))
    total = epochs * len(loader); warm = max(1, int(0.05 * total))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    best, best_state, bad, losses, vals = -1, None, 0, [], []
    t0 = time.time()
    use_amp = bool(CFG.get("use_amp")) and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for ep in range(epochs):
        model.train(); run = 0; t_ep = time.time()
        for ids, att, labels in loader:
            ids, att, labels = ids.to(device), att.to(device), labels.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(input_ids=ids, attention_mask=att, labels=labels).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        losses.append(run / len(loader)); t_train = time.time() - t_ep
        if not math.isfinite(losses[-1]):
            print("⚠️ La pérdida no es finita (nan/inf). Si usas use_amp=True, ponlo en False y vuelve a entrenar."); break
        t_v = time.time()
        r10 = evaluate(model, val_users, sid_tok, trie_, s2i_, n=CFG["val_users"])["Recall@10"]; vals.append(r10)
        if verbose: print(f"época {ep+1:2d}/{epochs} · loss {losses[-1]:.3f} · val Recall@10 {r10:.4f} · entrenar {t_train:.0f}s + validar {time.time()-t_v:.0f}s · esta época {time.time()-t_ep:.0f}s · acumulado {time.time()-t0:.0f}s")
        if r10 > best: best, best_state, bad = r10, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG["patience"]: print("early stopping"); break
    model.load_state_dict(best_state)
    return model, losses, vals

model, losses, vals = fit(sid_tok)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(losses); ax[0].set_title("loss de entrenamiento"); ax[0].set_xlabel("época")
ax[1].plot(vals); ax[1].set_title("Recall@10 en validación"); ax[1].set_xlabel("época"); plt.tight_layout(); plt.show()

## 7. Inferencia y evaluación en *test*
`recommend` usa `model.generate(...)` con `num_beams=20` y `prefix_allowed_tokens_fn` (el trie). Los 10 mejores beams se convierten de SID a ítem. Comparamos con un baseline de **popularidad** (los 10 ítems más frecuentes en entrenamiento).

Junto a cada Recall@10 mostramos su **error estándar** $\sqrt{p(1-p)/n}$: diferencias del orden de 1–2 errores estándar pueden deberse solo al azar (qué usuarios hay en test, semillas, etc.).

In [ ]:
def fmt(r, n=None):
    s = " · ".join(f"{k} {v:.4f}" for k, v in r.items())
    if n:
        p = r["Recall@10"]; s += f"   (±{math.sqrt(p * (1 - p) / n):.4f} e.e. de Recall@10)"
    return s

n_test = min(CFG["test_n"] or len(test_users), len(test_users))
t0 = time.time()
res = evaluate(model, test_users, sid_tok, trie, sid2item, n=CFG["test_n"])
print(f"GenRec (mini-TIGER) en test [{time.time()-t0:.0f}s]:", fmt(res, n_test))

pop = collections.Counter(i for s in train_seqs for i in s)
top_pop = [i for i, _ in pop.most_common(10)]
res_pop = ranking_metrics([top_pop] * len(test_users), [t for _, t in test_users])
print("Popularidad:", fmt(res_pop, len(test_users)))

In [ ]:
# Un ejemplo cualitativo
h, tgt = test_users[7]
rec, sc = recommend(model, [h], sid_tok, trie, sid2item, CFG["max_items"], beams=CFG["beams"], topk=CFG["topk"])
print("HISTORIA (últimos 5):")
for i in h[-5:]: print("  •", items[i]["title"][:90])
print("\nOBJETIVO REAL:", items[tgt]["title"][:90], sid_tok[tgt].tolist())
print("\nRECOMENDACIONES:")
for i, s in zip(rec[0], sc[0]):
    print(f"  {'✔' if i == tgt else ' '} {s:.3f}  {items[i]['title'][:90] if i >= 0 else '(SID inválido)'}")

# Parte B
## 8. 🧪 Ejercicio 2 — Cambiar el tokenizador: RQ-VAE vs RQ-KMeans
TIGER no usa RQ-KMeans, sino un **RQ-VAE** (*Residual-Quantized Variational AutoEncoder*). La cuantización residual es la misma de la sección 3, con dos diferencias:

1. **Se cuantiza un espacio latente aprendido.** Un *encoder* (MLP) lleva el embedding de 768 dimensiones a un vector $z$ de 32 dimensiones; ahí se aplica la cuantización residual, y un *decoder* intenta reconstruir el embedding original a partir de $\hat z = e^{(1)}_{c_1} + e^{(2)}_{c_2} + e^{(3)}_{c_3}$.
2. **Todo se entrena por gradiente**, encoder, decoder y codebooks a la vez, minimizando el error de reconstrucción más una pérdida que acerca los códigos a los residuos (*codebook*) y los residuos a los códigos (*commitment*, con peso $\beta$). Como el `argmin` no es derivable, se usa el truco *straight-through*.

Si el encoder y el decoder fueran la identidad y los codebooks se ajustaran con k-means, tendrías exactamente RQ-KMeans.

![Diagrama RQ-VAE](https://raw.githubusercontent.com/PUC-RecSys-Class/RecSysPUC-2026-2/master/practicos/RQ-VAE.png)

**El problema típico: colapso del codebook.** Si se inicializan al azar, muchos códigos nunca son el "más cercano" de ningún ítem, no reciben gradiente y quedan muertos. Así hay menos SIDs distintos, más colisiones y prefijos menos informativos. Nuestra implementación usa dos remedios de la literatura: **inicializar los codebooks con k-means** sobre el espacio latente (como TIGER) y **reasignar los códigos muertos** a residuos de ítems al azar durante el entrenamiento.

No tienes que programar nada: ejecuta las celdas y responde las preguntas al final de la sección.

### 8.1 Entrenar el RQ-VAE (< 1 min)

In [ ]:
import torch.nn.functional as F

RQVAE_CFG = dict(
    hidden=[512, 256, 128], latent=32,   # encoder 768→512→256→128→32 (decoder espejo), como TIGER
    beta=0.25,                           # peso de la pérdida de commitment
    lr=1e-3, batch=1024,
    ae_epochs=20,                        # calentamiento: solo autoencoder (sin cuantizar)
    epochs=300,                          # épocas con cuantización
    kmeans_init=True,                    # inicializar los codebooks con RQ-KMeans sobre z
    reset_dead=True,                     # reasignar códigos que nadie usa (evita colapso)
)

class RQVAE(nn.Module):
    def __init__(self, d_in, hidden, latent, levels, K, beta):
        super().__init__()
        dims = [d_in] + hidden + [latent]
        def mlp(ds):
            layers = []
            for a, b in zip(ds[:-1], ds[1:]):
                layers += [nn.Linear(a, b), nn.ReLU()]
            return nn.Sequential(*layers[:-1])          # sin ReLU en la última capa
        self.encoder, self.decoder = mlp(dims), mlp(dims[::-1])
        self.codebooks = nn.Parameter(torch.randn(levels, K, latent) * 0.01)
        self.beta = beta

    def quantize(self, z):
        # Cuantización residual: igual que RQ-KMeans, pero con codebooks aprendidos por gradiente.
        res, z_q, codes, loss = z, torch.zeros_like(z), [], 0.0
        for cb in self.codebooks:
            idx = torch.cdist(res, cb).argmin(1)
            e = cb[idx]
            loss = loss + F.mse_loss(e, res.detach()) + self.beta * F.mse_loss(res, e.detach())
            z_q, res = z_q + e, res - e.detach()
            codes.append(idx)
        z_q = z + (z_q - z).detach()                    # straight-through: el gradiente salta el argmin
        return z_q, torch.stack(codes, 1), loss

    def forward(self, x):
        z = self.encoder(x)
        z_q, codes, q_loss = self.quantize(z)
        return self.decoder(z_q), codes, q_loss

    @torch.no_grad()
    def partial_decode(self, x, levels):
        # Reconstrucción usando solo los primeros `levels` códigos (para ver el refinamiento grueso→fino).
        z = self.encoder(x)
        res, z_q = z, torch.zeros_like(z)
        for cb in self.codebooks[:levels]:
            e = cb[torch.cdist(res, cb).argmin(1)]
            z_q, res = z_q + e, res - e
        return self.decoder(z_q)


def train_rqvae(emb, levels, K, cfg=RQVAE_CFG, seed=0, verbose=True):
    torch.manual_seed(seed)
    x_raw = torch.tensor(emb, device=device)
    mu, sd = x_raw.mean(0), x_raw.std(0) + 1e-6        # estandarizamos cada dimensión
    x = (x_raw - mu) / sd
    model = RQVAE(x.size(1), cfg["hidden"], cfg["latent"], levels, K, cfg["beta"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    n, hist = x.size(0), collections.defaultdict(list)
    t0 = time.time()
    for ep in range(cfg["ae_epochs"] + cfg["epochs"]):
        quant = ep >= cfg["ae_epochs"]
        if quant and ep == cfg["ae_epochs"] and cfg["kmeans_init"]:
            with torch.no_grad():                      # codebooks iniciales = RQ-KMeans sobre el espacio latente
                res = model.encoder(x)
                for l in range(levels):
                    c, a = kmeans(res, K, seed=seed + l)
                    model.codebooks.data[l] = c
                    res = res - c[a]
        model.train(); perm = torch.randperm(n, device=device); rec_sum = q_sum = 0.0
        used = torch.zeros(levels, K, dtype=torch.bool, device=device)
        for s in range(0, n, cfg["batch"]):
            xb = x[perm[s:s + cfg["batch"]]]
            if quant:
                x_hat, codes_b, q_loss = model(xb)
                used[torch.arange(levels, device=device)[None, :].expand_as(codes_b), codes_b] = True
            else:
                x_hat, q_loss = model.decoder(model.encoder(xb)), torch.zeros((), device=device)
            rec = F.mse_loss(x_hat, xb)
            loss = rec + q_loss
            opt.zero_grad(); loss.backward(); opt.step()
            rec_sum += rec.item() * len(xb); q_sum += q_loss.item() * len(xb)
        hist["rec"].append(rec_sum / n); hist["q"].append(q_sum / n)
        if quant:
            hist["used"].append(used.sum(1).tolist())
            if cfg["reset_dead"] and ep < cfg["ae_epochs"] + cfg["epochs"] - 10:
                with torch.no_grad():                  # códigos muertos ← residuos de ítems al azar
                    res = model.encoder(x)
                    for l in range(levels):
                        cb = model.codebooks.data[l]
                        dead = (~used[l]).nonzero().squeeze(1)
                        if len(dead):
                            cb[dead] = res[torch.randint(0, n, (len(dead),), device=device)]
                        res = res - cb[torch.cdist(res, cb).argmin(1)]
        if verbose and (ep + 1) % 50 == 0:
            u = f" · códigos usados por nivel {hist['used'][-1]}" if quant else ""
            print(f"época {ep+1:3d} · recon {hist['rec'][-1]:.4f} · cuantización {hist['q'][-1]:.4f}{u} · {time.time()-t0:.0f}s")
    model.eval()
    with torch.no_grad():
        _, codes_v, _ = model(x)
        errs_v = []
        for l in range(1, levels + 1):                 # error relativo en el espacio ORIGINAL (comparable a RQ-KMeans)
            x_rec = model.partial_decode(x, l) * sd + mu
            errs_v.append(((x_rec - x_raw).norm(dim=1).mean() / x_raw.norm(dim=1).mean()).item())
    return model, codes_v.cpu().numpy(), errs_v, hist


t0 = time.time()
rqvae, codes_vae, errs_vae, hist_vae = train_rqvae(emb, CFG["levels"], K)
print(f"RQ-VAE entrenado en {time.time()-t0:.0f}s")
print("error relativo de reconstrucción tras cada nivel:", [f"{e:.3f}" for e in errs_vae],
      "  (RQ-KMeans:", [f"{e:.3f}" for e in errs], ")")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist_vae["rec"]); ax[0].axvline(RQVAE_CFG["ae_epochs"], ls="--", c="gray")
ax[0].set_yscale("log"); ax[0].set_title("pérdida de reconstrucción"); ax[0].set_xlabel("época")
used = np.array(hist_vae["used"])
for l in range(used.shape[1]): ax[1].plot(np.arange(len(used)) + RQVAE_CFG["ae_epochs"], used[:, l], label=f"nivel {l+1}")
ax[1].set_title(f"códigos usados (de {K})"); ax[1].set_xlabel("época"); ax[1].legend(); plt.tight_layout(); plt.show()

### 8.2 Comparar los tokenizadores
Antes de entrenar TIGER, podemos comparar la calidad de los Semantic IDs por sí mismos:

- **Error de reconstrucción:** qué tanto del embedding original se conserva con 3 códigos (en el espacio original de 768-d, igual para ambos métodos).
- **Colisiones:** ítems que comparten $(c_1,c_2,c_3)$.
- **Coherencia semántica:** similitud coseno promedio entre pares de ítems que comparten prefijo, comparada con pares al azar.

In [ ]:
def tokenizer_stats(codes3, errs, emb, K):
    # Métricas del tokenizador (sin entrenar ningún recomendador).
    e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    st = {"error reconstrucción (3 niveles)": errs[-1]}
    for l in range(codes3.shape[1]):
        p = np.bincount(codes3[:, l], minlength=K) / len(codes3)
        st[f"códigos usados nivel {l+1}"] = int((p > 0).sum())
        st[f"perplejidad nivel {l+1}"] = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
    cnt = collections.Counter(map(tuple, codes3))
    st["% ítems en colisión"] = 100 * sum(c for c in cnt.values() if c > 1) / len(codes3)
    st["máx. ítems por SID"] = max(cnt.values())
    # Coherencia semántica: similitud coseno promedio entre pares de ítems que comparten prefijo
    def coherence(keys):
        num = den = 0.0
        groups = collections.defaultdict(list)
        for i, k_ in enumerate(keys): groups[k_].append(i)
        for g in groups.values():
            if len(g) < 2: continue
            s = e[g].sum(0); n = len(g)
            num += (s @ s - n); den += n * (n - 1)
        return num / max(den, 1)
    s_all = e.sum(0); n = len(e)
    st["coseno pares al azar"] = float((s_all @ s_all - n) / (n * (n - 1)))
    st["coseno pares mismo c1"] = float(coherence([int(c) for c in codes3[:, 0]]))
    st["coseno pares mismo (c1,c2)"] = float(coherence(list(map(tuple, codes3[:, :2]))))
    return st

def tokenizer_report(name, codes3, errs, emb, K):
    st = tokenizer_stats(codes3, errs, emb, K)
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in st.items()})
    return st


sids_vae = dedup_token_solution(codes_vae)
if sids_vae[:, 3].max() >= K:
    raise RuntimeError("El RQ-VAE tiene demasiadas colisiones para K: revisa RQVAE_CFG (¿desactivaste kmeans_init/reset_dead?)")
st_km = tokenizer_stats(codes, errs, emb, K)
st_vae = tokenizer_stats(codes_vae, errs_vae, emb, K)
print(f"{'':34s}{'RQ-KMeans':>12s}{'RQ-VAE':>12s}")
for k in st_km:
    a, b = st_km[k], st_vae[k]
    f = (lambda v: f"{v:12.3f}") if isinstance(a, float) else (lambda v: f"{v:12d}")
    print(f"{k:34s}{f(a)}{f(b)}")

In [ ]:
# Mismo ítem q que en la sección 3: ¿con quién comparte prefijo según cada tokenizador?
for name, S_ in [("RQ-KMeans", sids), ("RQ-VAE", sids_vae)]:
    same = np.where((S_[:, 0] == S_[q, 0]) & (S_[:, 1] == S_[q, 1]))[0]
    print(f"\n{name} · ÍTEM: {items[q]['title'][:70]} {S_[q]} · {len(same)} ítems comparten (c1,c2)")
    for j in same[same != q][:5]: print("   ", items[j]["title"][:80], S_[j])

### 8.3 Entrenar TIGER con los SIDs del RQ-VAE (~10–15 min)
Usamos **exactamente** el mismo modelo, datos, hiperparámetros y evaluación que en la Parte A: lo único que cambia es el tokenizador.

In [ ]:
sid_tok_vae = codes_to_tokens(sids_vae, K)
trie_vae, sid2item_vae = build_trie(sid_tok_vae), make_sid2item(sid_tok_vae)
model_vae, losses_vae, vals_vae = fit(sid_tok_vae)
t0 = time.time()
res_vae = evaluate(model_vae, test_users, sid_tok_vae, trie_vae, sid2item_vae, n=CFG["test_n"])
print(f"TIGER + RQ-VAE en test [{time.time()-t0:.0f}s]:", fmt(res_vae, n_test))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(losses, label="RQ-KMeans"); ax[0].plot(losses_vae, label="RQ-VAE"); ax[0].set_title("loss de entrenamiento")
ax[1].plot(vals, label="RQ-KMeans"); ax[1].plot(vals_vae, label="RQ-VAE"); ax[1].set_title("Recall@10 en validación")
for a in ax: a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Tabla resumen (si ejecutaste el Ejercicio 4, también aparecen los SIDs aleatorios)
rows = [("Popularidad", res_pop), ("TIGER + RQ-KMeans", res), ("TIGER + RQ-VAE", res_vae)]
if "res_rand" in globals(): rows.append(("TIGER + SIDs aleatorios", res_rand))
print(f"{'':26s}" + "".join(f"{k:>11s}" for k in res))
for name, r in rows:
    print(f"{name:26s}" + "".join(f"{v:11.4f}" for v in r.values()))
p = res["Recall@10"]
print(f"\nerror estándar aprox. de Recall@10 con n={n_test}: ±{math.sqrt(p * (1 - p) / n_test):.4f}")

**Preguntas (responde en una celda de texto debajo; justifica con los números que obtuviste):**

1. **Reconstrucción.** ¿Cuál de los dos tokenizadores conserva mejor la información del embedding con 3 niveles? ¿Cómo cambia el error de un nivel al siguiente en cada uno? Da una razón por la que un RQ-VAE podría reconstruir mejor que RQ-KMeans y otra por la que podría hacerlo peor (piensa en la dimensión del espacio que se cuantiza y en qué se optimiza).
2. **Uso de códigos y colisiones.** Compara códigos usados y colisiones por nivel. Luego, en la celda de la sección 8.1, pon `RQVAE_CFG.update(kmeans_init=False, reset_dead=False)` antes de entrenar, vuelve a ejecutar 8.1 y 8.2 (no hace falta la 8.3) y describe qué pasa. ¿Por qué el colapso del codebook es un problema para un recomendador generativo? *(Al terminar, vuelve a poner ambas opciones en `True` y ejecuta de nuevo 8.1 y 8.2.)*
3. **Recomendación.** ¿Cuál obtiene mejor Recall@10 y NDCG@10 en test? Considerando el error estándar, ¿la diferencia es concluyente? ¿Una mejor reconstrucción se tradujo en una mejor recomendación? Explica por qué podría no hacerlo.
4. **Costo-beneficio.** Compara ambos métodos en tiempo de entrenamiento, número de hiperparámetros y riesgos (p. ej., colapso).

## 9. 🧪 Ejercicios de análisis
**Ejercicio 3 — ¿Para qué sirve el trie?** La celda siguiente genera **con** y **sin** restricción usando tres modelos: uno **sin entrenar** (pesos al azar), uno entrenado **una sola época** (~1 min) y el modelo final.
- Para cada uno: ¿qué porcentaje de las recomendaciones corresponde a SIDs que no existen? ¿Cómo cambia Recall@10 con y sin trie?
- Hay $256^3 \cdot 256 \approx 4\cdot10^9$ combinaciones posibles de 4 tokens, pero solo ~12 mil SIDs válidos. ¿Cómo explicas que el modelo entrenado casi nunca invente SIDs? Si casi no los inventa, ¿por qué igual conviene usar el trie en producción?
- ¿Por qué este problema es propio de los recomendadores generativos y no de los modelos de *ranking* clásicos?

In [ ]:
users_sub = random.Random(0).sample(test_users, min(1000, len(test_users)))
model_0ep = new_model(V)                              # pesos al azar, sin entrenar
model_1ep, _, _ = fit(sid_tok, epochs=1, verbose=False)
for name, m in [("modelo sin entrenar", model_0ep), ("modelo con 1 época", model_1ep), ("modelo final", model)]:
    rec_u, _ = recommend(m, [h for h, _ in users_sub], sid_tok, trie, sid2item, CFG["max_items"],
                         beams=CFG["beams"], topk=CFG["topk"], constrained=False)
    inval = np.mean([i == -1 for r in rec_u for i in r])
    r_u = ranking_metrics(rec_u, [t for _, t in users_sub])
    r_c = evaluate(m, users_sub, sid_tok, trie, sid2item)
    print(f"--- {name}: SIDs inválidos sin restricción: {100*inval:.1f}%")
    print("  Sin restricción:", fmt(r_u))
    print("  Con trie       :", fmt(r_c))

**Ejercicio 4 — ¿Importa la semántica del SID?** Entrena el mismo modelo con **SIDs aleatorios** (cada ítem recibe 3 códigos al azar + token de colisión) y compara. Si los SIDs semánticos ayudan, debería verse en Recall/NDCG. *(Usa pocas épocas si vas justo de tiempo: ~4 épocas (≈8 min) bastan para ver la tendencia; compara contra el curso de validación del modelo principal.)*

In [ ]:
rng = np.random.RandomState(0)
rand_codes = rng.randint(0, K, size=codes.shape)
rand_sids = dedup_token_solution(rand_codes)
assert rand_sids[:, 3].max() < K
rand_tok = codes_to_tokens(rand_sids, K)
model_rand, _, vals_rand = fit(rand_tok, epochs=min(4, CFG["epochs"]))
res_rand = evaluate(model_rand, test_users, rand_tok, build_trie(rand_tok), make_sid2item(rand_tok), n=CFG["test_n"])
print("SIDs aleatorios:", fmt(res_rand, n_test))
print("SIDs RQ-KMeans :", fmt(res, n_test))
if "res_vae" in globals(): print("SIDs RQ-VAE    :", fmt(res_vae, n_test))

**Ejercicio 5 — Experimenta (elige al menos uno) y reporta tus resultados en una tabla:**
- Cambia `CFG["K"]` (por ejemplo 64 o 128) y/o `CFG["levels"]` (2 o 4). Vuelve a ejecutar desde la sección 3. ¿Cómo cambian la tasa de colisiones y el Recall@10?
- Aumenta/disminuye `max_items` o el tamaño del modelo (`d_model`, `layers`).
- Cambia `beams` de 20 a 5 y a 50 en la evaluación. ¿Qué pasa con la calidad y el tiempo?
- Usa otro encoder de texto (por ejemplo `google/flan-t5-large`) en `CFG["emb_model"]`.
- Cambia hiperparámetros del RQ-VAE en `RQVAE_CFG` (`latent`, `beta`, `epochs`) y vuelve a ejecutar la sección 8.

**Preguntas de reflexión (responde en una celda de texto):**
1. ¿Qué ventaja tiene un SID jerárquico (grueso→fino) frente a un id numérico arbitrario cuando aparece un ítem nuevo (*cold-start*)?
2. TIGER hashea el id del usuario en tokens adicionales; aquí no lo hicimos. ¿Qué información pierde el modelo? ¿Cómo la agregarías?
3. Tanto RQ-KMeans como el RQ-VAE de la sección 8 se entrenan **antes y por separado** del recomendador, solo con el texto de los ítems. ¿Qué cambiaría si el tokenizador se entrenara **junto** al recomendador, usando también las interacciones (como en ETEGRec)? ¿Qué ventaja y qué riesgo ves?

## 10. Exportar artefactos para la Actividad 3 (solo para el/la docente)
Este paso guarda embeddings, SIDs, metadatos, secuencias y el modelo entrenado en `artifacts_genrec.zip`. La **Actividad 3** (inferencia, sin entrenar) carga ese archivo. Sube el zip a Google Drive, compártelo con enlace y copia el *id* del archivo.

In [ ]:
ART = "artifacts"
os.makedirs(ART, exist_ok=True)
model.save_pretrained(f"{ART}/model")
np.save(f"{ART}/sids.npy", sids.astype(np.int16))
np.save(f"{ART}/emb.npy", emb.astype(np.float16))
json.dump(items, open(f"{ART}/items.json", "w"))
json.dump(seqs, open(f"{ART}/seqs.json", "w"))
json.dump({"K": K, "D": D, "max_items": CFG["max_items"], "levels": CFG["levels"], "emb_model": CFG["emb_model"],
           "test_recall@10": res["Recall@10"], "test_ndcg@10": res["NDCG@10"]}, open(f"{ART}/config.json", "w"))
import shutil
shutil.make_archive("artifacts_genrec", "zip", ART)
print("listo:", os.path.getsize("artifacts_genrec.zip") / 1e6, "MB")
try:
    from google.colab import files; files.download("artifacts_genrec.zip")
except Exception: pass

## Resumen
Construiste el pipeline completo de un recomendador generativo: **texto → embedding → Semantic ID (RQ-KMeans o RQ-VAE + token anti-colisión) → Transformer encoder-decoder → generación con beam search restringido → Recall/NDCG**. Es la receta de TIGER; GRID la implementa a escala con distintos tokenizadores (RQ-KMeans, RVQ, RQ-VAE) y flan-t5-xl.

**Referencias:** Rajput et al., *Recommender Systems with Generative Retrieval (TIGER)*, NeurIPS 2023 · Ju et al., *Generative Recommendation with Semantic IDs: A Practitioner's Handbook (GRID)*, 2025 · Singh et al., *Better Generalization with Semantic IDs*, RecSys 2024 · Zhou et al., *OneRec Technical Report*, 2025.